In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
df = pd.read_csv("Customer_Churn_Dataset_Surendra.csv")

print("=== Dataset Shape ===")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

print("\n=== First 5 Rows ===")
print(df.head())

print("\n=== Data Types ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Rows ===")
print("Duplicates found:", df.duplicated().sum())

print("\n=== Churn Count ===")
print(df["Churn"].value_counts())

print("\n=== Churn Percentage ===")
print(df["Churn"].value_counts(normalize=True).mul(100).round(2))

=== Dataset Shape ===
Rows: 7043 | Columns: 21

=== First 5 Rows ===
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               

In [3]:
df = df.rename(columns={
    "customerID"      : "CustomerID",
    "gender"          : "Gender",
    "tenure"          : "Tenure",
    "Contract"        : "ContractType",
    "PaymentMethod"   : "PaymentMethod",
    "InternetService" : "InternetService"
})

# Add Age and SupportTickets (not in Telco dataset)
np.random.seed(42)
df["Age"] = np.random.randint(18, 65, len(df))
df["SupportTickets"] = np.random.randint(0, 10, len(df))

# Fix TotalCharges — stored as string in Telco dataset
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Standardize Churn to Yes/No
df["Churn"] = df["Churn"].map({"Yes": "Yes", "No": "No", 1: "Yes", 0: "No"})

print("Columns after preparation:")
print(df.columns.tolist())
print("\nMissing values after prep:")
print(df.isnull().sum())

Columns after preparation:
['CustomerID', 'Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'ContractType', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Age', 'SupportTickets']

Missing values after prep:
CustomerID           0
Gender               0
SeniorCitizen        0
Partner              0
Dependents           0
Tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
ContractType         0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
Age                  0
SupportTickets       0
dtype: int64


In [4]:
df = df.drop("CustomerID", axis=1)

print("CustomerID removed.")
print("Remaining columns:", df.columns.tolist())

CustomerID removed.
Remaining columns: ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'ContractType', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Age', 'SupportTickets']


In [5]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())
print("\nChurn distribution:")
print(y.value_counts())

Features (X) shape: (7043, 21)
Target (y) shape: (7043,)

Feature columns: ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'ContractType', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Age', 'SupportTickets']

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=== Train-Test Split (80:20) ===")
print("Total rows         :", len(df))
print("Training rows (80%):", X_train.shape[0])
print("Testing rows  (20%):", X_test.shape[0])

print("\nTrain Churn ratio:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest Churn ratio:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

=== Train-Test Split (80:20) ===
Total rows         : 7043
Training rows (80%): 5634
Testing rows  (20%): 1409

Train Churn ratio:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

Test Churn ratio:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


In [7]:
numeric_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportTickets"
]

categorical_features = [
    "Gender",
    "ContractType",
    "PaymentMethod",
    "InternetService"
]

print("Numerical features   :", numeric_features)
print("Categorical features :", categorical_features)

Numerical features   : ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportTickets']
Categorical features : ['Gender', 'ContractType', 'PaymentMethod', 'InternetService']


In [8]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline built successfully")

Numerical pipeline built successfully


In [9]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical pipeline built successfully")

Categorical pipeline built successfully


In [10]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("ColumnTransformer created successfully")

ColumnTransformer created successfully


In [11]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

print("Final reusable ML pipeline:")
print(model_pipeline)

Final reusable ML pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges',
                                                   'SupportTickets']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                  

In [12]:
model_pipeline.fit(X_train, y_train)

print("Pipeline trained successfully on", X_train.shape[0], "rows")

Pipeline trained successfully on 5634 rows


In [13]:
y_pred = model_pipeline.predict(X_test)

print("Predictions made on", X_test.shape[0], "test rows")
print("Sample predictions:", y_pred[:10])

Predictions made on 1409 test rows
Sample predictions: ['No' 'Yes' 'No' 'Yes' 'No' 'No' 'No' 'No' 'No' 'Yes']


In [14]:
print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)

print("\nAccuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\nHow to read:")
print("  True Negative  (top-left) :", cm[0][0], "→ correctly predicted No Churn")
print("  False Positive (top-right):", cm[0][1], "→ predicted Churn but actually No Churn")
print("  False Negative (bot-left) :", cm[1][0], "→ predicted No Churn but actually Churned")
print("  True Positive  (bot-right):", cm[1][1], "→ correctly predicted Churn")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

MODEL EVALUATION

Accuracy: 77.5 %

Confusion Matrix:
[[904 131]
 [186 188]]

How to read:
  True Negative  (top-left) : 904 → correctly predicted No Churn
  False Positive (top-right): 131 → predicted Churn but actually No Churn
  False Negative (bot-left) : 186 → predicted No Churn but actually Churned
  True Positive  (bot-right): 188 → correctly predicted Churn

Classification Report:
              precision    recall  f1-score   support

          No       0.83      0.87      0.85      1035
         Yes       0.59      0.50      0.54       374

    accuracy                           0.78      1409
   macro avg       0.71      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [15]:
joblib.dump(model_pipeline, "customer_churn_pipeline.pkl")
print("Pipeline saved as customer_churn_pipeline.pkl")

Pipeline saved as customer_churn_pipeline.pkl


In [16]:
loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")
print("Pipeline loaded successfully")

Pipeline loaded successfully


In [17]:
new_customer = pd.DataFrame({
    "Gender"         : ["Female"],
    "Age"            : [32],
    "Tenure"         : [5],
    "MonthlyCharges" : [850],
    "TotalCharges"   : [4250],
    "ContractType"   : ["Monthly"],
    "PaymentMethod"  : ["Credit Card"],
    "InternetService": ["Fiber"],
    "SupportTickets" : [3]
})

prediction = loaded_pipeline.predict(new_customer)
probability = loaded_pipeline.predict_proba(new_customer)

print("=== New Customer Churn Prediction ===")
print("Churn Prediction :", prediction[0])
print("Churn Probability:", round(probability[0][1] * 100, 2), "%")

=== New Customer Churn Prediction ===
Churn Prediction : No
Churn Probability: 32.0 %
